# 05 · Lista de casa: quem são os clientes encantados?

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

Na aula construímos um modelo para encontrar clientes insatisfeitos. Agora você vai construir, sozinho, um modelo para o problema oposto, tomando as decisões que tomamos juntos em sala.

## O cenário

O time de marketing quer convidar para um programa de embaixadores os clientes **encantados** com a compra: aqueles que dariam **nota 5**. O convite será enviado a pessoas que escreveram sobre a compra em canais onde **não existe nota nem a pergunta de recomendação** (e-mail, chat, redes sociais). O modelo, portanto, só pode usar o texto, o perfil do cliente e as informações do produto.

## Instruções

- Trabalhe em uma cópia deste notebook. Substitua cada `...` pelo seu código e escreva as respostas nos espaços indicados.
- Você pode (e deve) consultar o notebook da aula. O objetivo é que você saiba **adaptar** o que foi feito, e não reproduzir de memória.
- As células ✅ fazem conferências básicas de formato. Elas **não** garantem que a resposta esteja correta.
- **Entrega:** o notebook executado de ponta a ponta, com todas as saídas visíveis, no prazo e pelo canal combinados em aula.
- **Assistentes de IA** são permitidos como apoio. Na questão 7 você deve declarar como os utilizou. Você precisa ser capaz de explicar oralmente qualquer linha entregue.

| Questão | Tema | Pontos |
|---|---|---|
| 1 | Divisão e referência ingênua | 1,0 |
| 2 | Pipeline tabular e validação cruzada | 2,0 |
| 3 | Texto no pipeline | 1,5 |
| 4 | Caça ao vazamento | 2,0 |
| 5 | Validação temporal | 1,5 |
| 6 | Limiar por resultado financeiro | 1,5 |
| 7 | Reflexão e declaração de uso de IA | 0,5 |

## Preparação (não altere)

As células abaixo carregam os dados e criam as mesmas variáveis derivadas da aula. O alvo agora é `nota_5`.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

print("scikit-learn", sklearn.__version__, "· pandas", pd.__version__)

SEMENTE = 42
URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

df = pd.read_csv(ARQUIVO, low_memory=False, dtype={"product_id": str})
df = df.dropna(subset=["recommend_to_a_friend"]).copy()

df["submission_date"] = pd.to_datetime(df["submission_date"])
idade = 2018 - df["reviewer_birth_year"]
df["idade"] = idade.where(idade.between(14, 100))
df["hora"] = df["submission_date"].dt.hour
df["dia_semana"] = df["submission_date"].dt.dayofweek.map(
    {0: "seg", 1: "ter", 2: "qua", 3: "qui", 4: "sex", 5: "sab", 6: "dom"}
)
df["texto"] = df["review_title"].fillna("") + ". " + df["review_text"].fillna("")

ALVO = "nota_5"
df[ALVO] = (df["overall_rating"] == 5).astype(int)

NUMERICAS = ["idade", "hora"]
CATEGORICAS = ["site_category_lv1", "site_category_lv2", "reviewer_state", "reviewer_gender", "dia_semana"]

print(df.shape)
df[[ALVO]].value_counts(normalize=True).round(3)

## Questão 1 (1,0 ponto): divisão e referência ingênua

a) Monte `X` e `y`. Em `X`, deixe de fora o alvo **e toda coluna que não estará disponível no cenário de uso** descrito acima.

b) Separe 20% para teste, com estratificação e `random_state=SEMENTE`, nas variáveis `X_treino`, `X_teste`, `y_treino`, `y_teste`.

c) Calcule `acuracia_ingenua`: a acurácia, no teste, de um modelo que sempre responde a classe mais frequente.

In [ ]:
# ✏️ Questão 1
# SEU CÓDIGO AQUI
X = ...
y = ...
X_treino, X_teste, y_treino, y_teste = ...
acuracia_ingenua = ...

In [ ]:
# ✅ Conferência de formato da questão 1 (não altere)
assert len(X_treino) + len(X_teste) == len(df) and abs(len(X_teste) / len(df) - 0.2) < 0.001
assert ALVO not in X.columns, "o alvo não pode estar em X"
assert 0.5 < acuracia_ingenua < 0.8
print(f"Treino: {len(X_treino)} · teste: {len(X_teste)} · acurácia ingênua: {acuracia_ingenua:.3f}")

**1d)** Quais colunas você retirou de `X` além do alvo, e por quê? Qual seria a precisão média (*average precision*) de um modelo que chuta ao acaso neste problema?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 2 (2,0 pontos): pipeline tabular e validação cruzada

Construa `modelo_tabular`, um `Pipeline` com:

- um `ColumnTransformer` que trate as colunas de `NUMERICAS` (imputação e padronização) e as de `CATEGORICAS` (imputação e one-hot, com tratamento para categorias raras e desconhecidas);
- uma `LogisticRegression`.

Avalie com validação cruzada estratificada de 5 dobras **no treino** e guarde a média da ROC AUC em `auc_tabular` e a média da precisão média em `ap_tabular`.

In [ ]:
# ✏️ Questão 2
# SEU CÓDIGO AQUI
modelo_tabular = ...
auc_tabular = ...
ap_tabular = ...

In [ ]:
# ✅ Conferência de formato da questão 2 (não altere)
from sklearn.pipeline import Pipeline as _Pipeline

assert isinstance(modelo_tabular, _Pipeline), "modelo_tabular deve ser um Pipeline"
assert 0.5 < auc_tabular < 0.9 and 0.3 < ap_tabular < 0.9
print(f"ROC AUC tabular: {auc_tabular:.3f} · precisão média tabular: {ap_tabular:.3f}")

**2b)** Explique, com suas palavras, o que acontece com o `SimpleImputer` e com o `OneHotEncoder` **em cada uma das 5 dobras** quando você passa o pipeline inteiro para a validação cruzada. Por que isso é importante?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 3 (1,5 ponto): texto no pipeline

Crie `modelo_texto`, acrescentando ao pré-processamento da questão 2 um `TfidfVectorizer` sobre a coluna `texto`. Avalie da mesma forma e guarde a média da ROC AUC em `auc_texto` e a da precisão média em `ap_texto`.

In [ ]:
# ✏️ Questão 3
# SEU CÓDIGO AQUI
modelo_texto = ...
auc_texto = ...
ap_texto = ...

In [ ]:
# ✅ Conferência de formato da questão 3 (não altere)
assert isinstance(modelo_texto, _Pipeline)
assert auc_texto > auc_tabular, "esperava-se que o texto melhorasse o modelo"
print(f"ROC AUC com texto: {auc_texto:.3f} (tabular: {auc_tabular:.3f})")

**3b)** Na aula, o modelo com texto chegou a uma ROC AUC perto de 0,96 para "não recomenda". Aqui o resultado para "nota 5" é mais baixo. Proponha uma explicação e descreva um experimento simples que ajudaria a confirmá-la.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 4 (2,0 pontos): caça ao vazamento

Um colega entregou o código abaixo, animado com o resultado. Ele roda sem erro e tem **três vazamentos**.

In [ ]:
# Código do colega (não altere)
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

dados = df.copy()
dados["recomenda"] = (dados["recommend_to_a_friend"] == "Yes").astype(int)                       # linha A
dados["nota_media_cliente"] = dados.groupby("reviewer_id")["overall_rating"].transform("mean")   # linha B

vetorizador = TfidfVectorizer(min_df=5)
matriz_texto = vetorizador.fit_transform(dados["texto"])                                         # linha C

X_colega = hstack([matriz_texto, csr_matrix(dados[["recomenda", "nota_media_cliente"]].to_numpy(dtype=float))]).tocsr()
auc_colega = cross_val_score(
    LogisticRegression(max_iter=1000), X_colega, dados[ALVO], cv=5, scoring="roc_auc"
).mean()
print(f"ROC AUC do colega: {auc_colega:.3f}")

**4a)** Para cada uma das linhas A, B e C, explique qual é o vazamento e classifique-o em um dos três tipos vistos em aula (pré-processamento fora da validação, alvo disfarçado, estrutura ignorada). Diga também qual dos três você espera que seja responsável pela **maior parte** da diferença no número final, e por quê.

✏️ **Sua resposta:**

_Escreva aqui._

**4b)** Reescreva o experimento sem vazamentos, em `modelo_corrigido`, aproveitando as ideias legítimas do colega: usar o texto e usar o histórico do cliente. Avalie com validação cruzada de 5 dobras no treino e guarde a média da ROC AUC em `auc_corrigida`.

In [ ]:
# ✏️ Questão 4b
# SEU CÓDIGO AQUI
modelo_corrigido = ...
auc_corrigida = ...

In [ ]:
# ✅ Conferência de formato da questão 4 (não altere)
assert isinstance(modelo_corrigido, _Pipeline)
assert auc_corrigida < auc_colega, "o resultado honesto deveria ser menor que o do colega"
print(f"ROC AUC do colega: {auc_colega:.3f} · corrigida: {auc_corrigida:.3f}")

**4c)** Compare `auc_corrigida` com `auc_texto` (questão 3). Quanto do ganho que o colega atribuiu ao histórico do cliente era real? Escreva, em três ou quatro linhas, a mensagem que você enviaria a ele explicando por que o resultado não vai se repetir em produção.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 5 (1,5 ponto): validação temporal

Em produção, o modelo é treinado com avaliações antigas e aplicado às que chegam depois. A validação cruzada com sorteio não simula isso.

Esta questão usa uma divisão própria, por data, sobre o `df` completo:

a) Monte `treino_tempo` com as avaliações enviadas **antes de 1º de maio de 2018** e `teste_tempo` com as de **maio**.

b) Ajuste um modelo igual ao `modelo_texto` (use `sklearn.base.clone`) em `treino_tempo` e calcule `auc_temporal`, a ROC AUC em `teste_tempo`.

In [ ]:
# ✏️ Questão 5
# SEU CÓDIGO AQUI
treino_tempo = ...
teste_tempo = ...
auc_temporal = ...

In [ ]:
# ✅ Conferência de formato da questão 5 (não altere)
assert treino_tempo["submission_date"].max() < teste_tempo["submission_date"].min(), "há sobreposição de datas entre treino e teste"
assert len(treino_tempo) + len(teste_tempo) == len(df)
assert 0.5 < auc_temporal <= 1.0
print(f"ROC AUC temporal: {auc_temporal:.3f} · com sorteio: {auc_texto:.3f}")

**5b)** Compare `auc_temporal` com `auc_texto`. A diferença é relevante? O que o resultado sugere sobre a estabilidade do problema ao longo desses meses? Cite uma situação concreta, neste negócio, em que você esperaria uma diferença grande entre as duas formas de validar.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 6 (1,5 ponto): limiar por resultado financeiro

O marketing informou:

- cada convite enviado custa **R$ 2**;
- cada cliente encantado (nota 5) que recebe o convite gera, em média, **R$ 10** de retorno;
- convidar um cliente que não está encantado não gera retorno.

O **resultado** de uma política de convites é, portanto, `10 × (convidados encantados) − 2 × (total de convidados)`.

a) Obtenha, com `cross_val_predict`, as probabilidades de validação cruzada do `modelo_texto` para todas as linhas do treino, em `proba_treino`.

b) Entre os limiares de `np.linspace(0.05, 0.95, 19)`, encontre `limiar_q6`, o que **maximiza** o resultado no treino.

c) Só então ajuste o modelo em todo o treino, aplique `limiar_q6` ao **teste** e calcule `resultado_teste`, em reais.

In [ ]:
# ✏️ Questão 6
# SEU CÓDIGO AQUI
proba_treino = ...
limiar_q6 = ...
resultado_teste = ...

In [ ]:
# ✅ Conferência de formato da questão 6 (não altere)
assert len(proba_treino) == len(X_treino) and 0 <= float(np.min(proba_treino)) and float(np.max(proba_treino)) <= 1
assert 0.05 <= limiar_q6 <= 0.95
assert resultado_teste > 0, "com um limiar razoável, o resultado no teste deveria ser positivo"
print(f"Limiar: {limiar_q6:.2f} · resultado no teste: R$ " + f"{resultado_teste:,.0f}".replace(",", "."))

**6b)** O limiar encontrado ficou acima ou abaixo de 0,5? Explique por quê, usando os valores de custo e retorno. Compare o resultado do modelo com o de convidar todos os clientes.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 7 (0,5 ponto): reflexão

**7a)** Antes de colocar o `modelo_texto` em produção, cite **duas** verificações que você faria e que não foram feitas nesta lista.

**7b)** Declare como você usou assistentes de IA nesta lista (não usei / tirei dúvidas de conceito / gerei trechos de código / outro) e, se usou, descreva um momento em que a sugestão recebida estava errada ou precisou ser corrigida.

✏️ **Sua resposta:**

_Escreva aqui._